<a href="https://colab.research.google.com/github/Thorballer/Alain-Cycling/blob/main/better_midi_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("stareven233/maestrov300-hdf5")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'maestrov300-hdf5' dataset.
Path to dataset files: /kaggle/input/maestrov300-hdf5


In [ ]:
!pip install pretty-midi
!pip install note-seq
!pip install mir-eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.4/209.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.1 MB/s eta 0:00:00


In [ ]:
import os
import librosa
import pretty_midi
import torch
import pandas as pd
import h5py
import note_seq
import numpy as np
import torch.nn as nn
import math
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, DistributedSampler
import torch.optim as optim
import torch.nn.functional as F
import torchaudio.transforms as T

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
import os
import h5py
import torch
import torchaudio
import note_seq
import gc
from tqdm import tqdm

input_h5_dir = '/kaggle/input/maestrov300-hdf5'  # Update if your Colab path is different!

# --- NEW FRAME-LEVEL CONFIGURATION ---
SAMPLE_RATE = 16000
HOP_LENGTH = 512
FPS = SAMPLE_RATE / HOP_LENGTH  # 31.25 Hz

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=2048,
    hop_length=HOP_LENGTH,
    n_mels=229,
    f_min=30.0,
    f_max=8000.0
)

# 🚀 THE RAM VAULT
global_ram_cache = []

print("⚙️ Processing HDF5 directly into System RAM...")

track_counter = 0

for root, dirs, files in os.walk(input_h5_dir):
    for file_name in files:
        if not file_name.endswith('.h5'): continue

        file_path = os.path.join(root, file_name)
        with h5py.File(file_path, "r") as f:
            for track_name in tqdm(f.keys(), desc=f"Processing {file_name}"):

                # 1. AUDIO TO LOG-MEL
                full_audio = f[track_name]['audio'][:].astype('float32') / 32768.0
                waveform = torch.tensor(full_audio).unsqueeze(0)

                mel_spec = mel_transform(waveform)
                log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5)).squeeze(0)

                T_frames = log_mel_spec.shape[1]

                # 2. MIDI TO 88-KEY PIANO ROLLS
                raw_midi_bytes = f[track_name]['midi'][()].tobytes()
                ns = note_seq.NoteSequence.FromString(raw_midi_bytes)

                frames = torch.zeros((88, T_frames), dtype=torch.float32)
                onsets = torch.zeros((88, T_frames), dtype=torch.float32)
                velocities = torch.zeros((88, T_frames), dtype=torch.float32)

                for note in ns.notes:
                    pitch_idx = note.pitch - 21
                    if pitch_idx < 0 or pitch_idx >= 88:
                        continue

                    start_frame = int(note.start_time * FPS)
                    end_frame = int(note.end_time * FPS)

                    start_frame = min(start_frame, T_frames - 1)
                    end_frame = min(end_frame, T_frames)

                    if start_frame >= end_frame:
                        continue

                    frames[pitch_idx, start_frame:end_frame] = 1.0
                    onsets[pitch_idx, start_frame] = 1.0
                    velocities[pitch_idx, start_frame:end_frame] = note.velocity / 127.0

                # 3. COMPRESS AND APPEND TO RAM
                # Casting to float16 and bool saves massive amounts of system memory
                global_ram_cache.append({
                    "mel": log_mel_spec.to(torch.float16),
                    "frames": frames.to(torch.bool),
                    "onsets": onsets.to(torch.bool),
                    "velocities": velocities.to(torch.float16)
                })
                track_counter += 1

# Force garbage collection to free up temporary loading memory
gc.collect()
print(f"✅ Finished! Loaded {track_counter} full tracks directly into RAM.")

⚙️ Processing HDF5 directly into System RAM...


Processing 2015.h5: 100%|██████████| 129/129 [00:36<00:00,  3.55it/s]

✅ Finished! Loaded 1276 full tracks directly into RAM.


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class MaestroDirectRAMDataset(Dataset):
    def __init__(self, ram_cache, chunk_frames=512):
        self.chunk_frames = chunk_frames
        self.chunks = []
        self.ram_cache = ram_cache

        print("Pre-calculating 512-frame chunk indices...")
        for track_idx, data in enumerate(self.ram_cache):
            T_frames = data['mel'].shape[1]
            step_size = chunk_frames // 2

            for start in range(0, T_frames - chunk_frames, step_size):
                self.chunks.append({
                    'track_idx': track_idx,
                    'start': start,
                    'end': start + chunk_frames
                })

        print(f"✅ Ready! Created {len(self.chunks)} training chunks.")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk_info = self.chunks[idx]

        # Pull directly from the global RAM list
        data = self.ram_cache[chunk_info['track_idx']]
        s, e = chunk_info['start'], chunk_info['end']

        # Slice and cast back to float32 on the fly for the neural network
        mel = data['mel'][:, s:e].unsqueeze(0).to(torch.float32)
        frames = data['frames'][:, s:e].to(torch.float32)
        onsets = data['onsets'][:, s:e].to(torch.float32)
        vels = data['velocities'][:, s:e].to(torch.float32)

        # (You can insert your SpecAugment and Random Gain logic right here!)

        return mel, frames, onsets, vels

# Initialize the Dataset


In [ ]:
mid_length = int(len(global_ram_cache) * .05)

In [ ]:
test_dataset = MaestroDirectRAMDataset(global_ram_cache[:mid_length])
dataset = MaestroDirectRAMDataset(global_ram_cache[mid_length:len(global_ram_cache)])

Pre-calculating 512-frame chunk indices...
✅ Ready! Created 6081 training chunks.
Pre-calculating 512-frame chunk indices...
✅ Ready! Created 79563 training chunks.


In [ ]:
# --- INITIALIZE THE CLEAN DATALOADER ---
#dataset = MaestroFrameDataset()

# The perfect DataLoader for an entirely RAM-based dataset
loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,         # CRITICAL: Do not duplicate the RAM list!
    pin_memory=True        # Speeds up the final jump from CPU RAM to GPU VRAM
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,         # CRITICAL: Do not duplicate the RAM list!
    pin_memory=True        # Speeds up the final jump from CPU RAM to GPU VRAM
)
# Quick sanity check
mel, frames, onsets, vels = next(iter(loader))
print(f"Mel shape: {mel.shape}")       # Should be [8, 1, 229, 512]
print(f"Frames shape: {frames.shape}") # Should be [8, 88, 512]

Mel shape: torch.Size([16, 1, 229, 512])
Frames shape: torch.Size([16, 88, 512])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BidirectionalAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, num_kv_heads=None, dropout=0.1):
        super().__init__()

        self.num_heads = num_heads
        # Default to standard Multi-Head Attention if kv_heads isn't specified,
        # but supports GQA if you want to experiment with it.
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads
        self.num_queries_per_kv = self.num_heads // self.num_kv_heads

        self.head_dim = d_model // num_heads
        self.d_model = d_model
        self.dropout_p = dropout

        # Projections
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        kv_dim = self.num_kv_heads * self.head_dim
        self.k_proj = nn.Linear(d_model, kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        """
        Input: x shaped (Batch, Seq_Len, d_model) -> e.g., (8, 512, 512)
        Output: contextualized x shaped (Batch, Seq_Len, d_model)
        """
        B, T, _ = x.shape

        # 1. Linear Projections & Reshape for Attention
        # Shape: (B, T, Num_Heads, Head_Dim) -> Transpose to (B, Num_Heads, T, Head_Dim)
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # 2. Grouped-Query Attention (GQA) Broadcasting
        # If using standard MHA, this does nothing. If using GQA, it duplicates KV heads to match Q heads.
        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=1)

        # 3. Flash Attention / Memory Efficient Attention
        # Notice: is_causal=False, attn_mask=None.
        # The model sees the entire 512-frame chunk at once in both directions.
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False
        )

        # 4. Reshape and Project Out
        # Shape: (B, Num_Heads, T, Head_Dim) -> (B, T, Num_Heads, Head_Dim) -> (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

        return self.out_proj(out)

In [ ]:
class MelFrontend(nn.Module):
    def __init__(self, d_model=512, n_mels=229):
        super().__init__()

        # Input: [Batch, 1, 229 (freq), T (time)]
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 48, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            # Pool frequency by 2, leave time alone
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Dropout2d(0.1),

            # Block 2
            nn.Conv2d(48, 96, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            # Pool frequency by 2, leave time alone
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Dropout2d(0.1),

            # Block 3
            nn.Conv2d(96, 96, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.Dropout2d(0.1),
        )

        # Calculate frequency bins after two 2x poolings: 229 // 2 // 2 = 57
        cnn_out_channels = 96
        cnn_freq_bins = 57
        self.input_proj = nn.Linear(cnn_out_channels * cnn_freq_bins, d_model)

    def forward(self, x):
        # x is [Batch, 1, 229, T]
        x = self.cnn(x) # [Batch, 96, 57, T]

        B, C, F_bins, T = x.shape
        x = x.permute(0, 3, 1, 2).contiguous() # [Batch, T, 96, 57]
        x = x.view(B, T, C * F_bins)           # [Batch, T, 5472]

        return self.input_proj(x)              # [Batch, T, d_model]

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dff, dropout=0.1):
        super().__init__()
        # We need 2/3 of the standard dff to keep FLOPs constant
        # We multiply by 2 here because we split this layer into two (Gate & Value)
        self.w12 = nn.Linear(d_model, 2 * dff)
        self.w3 = nn.Linear(dff, d_model)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        residual = x
        x = self.norm(x)

        # 1. Project to 2x hidden size
        gate_plus_value = self.w12(x)

        # 2. Split into the Gate and the Value path
        gate, value = gate_plus_value.chunk(2, dim=-1)

        # 3. SwiGLU Math: SiLU(gate) * value
        # F.silu is the same as the 'Swish' activation
        x = F.silu(gate) * value

        # 4. Project back and add residual
        x = self.w3(x)
        return residual + self.dropout(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# (Keep the MelFrontend and BidirectionalAttention from the previous step)

class ConformerBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, kernel_size=31, dropout=0.1):
        super().__init__()

        # 1. Macaron Feed-Forward 1 (Half-step residual)
        self.ff1 = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_ff),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

        # 2. Multi-Head Self-Attention (Using our SDPA implementation)
        self.norm_attn = nn.LayerNorm(d_model)
        self.attn = BidirectionalAttention(d_model, n_heads, dropout=dropout)

        # 3. Convolution Module
        self.norm_conv = nn.LayerNorm(d_model)
        self.conv_expand = nn.Linear(d_model, d_model * 2)

        # Depthwise conv over the time dimension
        self.conv_depthwise = nn.Conv1d(
            d_model, d_model, kernel_size=kernel_size,
            padding=kernel_size // 2, groups=d_model
        )
        self.conv_norm = nn.BatchNorm1d(d_model)
        self.conv_act = nn.SiLU()
        self.conv_proj = nn.Linear(d_model, d_model)
        self.conv_drop = nn.Dropout(dropout)

        # 4. Macaron Feed-Forward 2 (Half-step residual)
        self.ff2 = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_ff),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

        self.norm_out = nn.LayerNorm(d_model)

    def _apply_conv(self, x):
        # Pointwise expand & GLU
        x = self.conv_expand(x)            # (batch, T, d_model * 2)
        x = F.glu(x, dim=-1)               # (batch, T, d_model)

        # Prep for 1D Convolution: (batch, channels, time)
        x = x.transpose(1, 2)              # (batch, d_model, T)

        # Depthwise Conv -> Norm -> Act
        x = self.conv_depthwise(x)
        x = self.conv_norm(x)
        x = self.conv_act(x)

        # Prep for Pointwise Project: (batch, time, channels)
        x = x.transpose(1, 2)              # (batch, T, d_model)

        # Pointwise project & Dropout
        x = self.conv_proj(x)
        return self.conv_drop(x)

    def forward(self, x):
        # Macaron architecture dictates 0.5 weights on the FFN residuals
        x = x + 0.5 * self.ff1(x)
        x = x + self.attn(self.norm_attn(x))
        x = x + self._apply_conv(self.norm_conv(x))
        x = x + 0.5 * self.ff2(x)
        return self.norm_out(x)

In [ ]:
class ConformerPianoTranscriptionModel(nn.Module):
    def __init__(
        self,
        d_model=512,
        num_heads=8,
        num_layers=6,
        d_ff=2048,
        kernel_size=31,  # 31 is standard, captures ~1 second of audio context per layer
        dropout=0.1
    ):
        super().__init__()

        self.frontend = MelFrontend(d_model=d_model)

        self.pos_dropout = nn.Dropout(dropout)
        self.register_buffer('pe', self._get_sinusoid_encoding(4096, d_model))

        # --- The Conformer Stack ---
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, num_heads, d_ff, kernel_size, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)

        # Output Classification Heads
        self.onset_head = nn.Linear(d_model, 88)
        self.frame_head = nn.Linear(d_model, 88)
        self.velocity_head = nn.Sequential(
            nn.Linear(d_model, 88),
            nn.Sigmoid()
        )

    def _get_sinusoid_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, mel):
        B, _, _, T = mel.shape

        x = self.frontend(mel)
        x = x + self.pe[:, :T, :]
        x = self.pos_dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.final_norm(x)

        onset_logits = self.onset_head(x)
        frame_logits = self.frame_head(x)
        velocity_pred = self.velocity_head(x)

        return onset_logits, frame_logits, velocity_pred

In [ ]:
d_model = 512
dff = 2048
num_heads = 8
dropout_rate = 0.1
num_layers=6

In [ ]:
model = ConformerPianoTranscriptionModel(
    d_model=d_model,
    num_heads=num_heads,
    d_ff=dff,
    num_layers=num_layers,
    dropout=dropout_rate
)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def roll_to_notes(onset_probs, frame_probs, velocity_preds, onset_threshold=0.3, frame_threshold=0.3, hop_length=512, sr=16000):
    """Converts 88-key binary piano rolls back into a list of MIDI note events."""
    frame_rate = sr / hop_length  # 31.25 Hz
    T = onset_probs.shape[0]
    notes = []

    for key in range(88):
        active = frame_probs[:, key] > frame_threshold
        onset  = onset_probs[:, key] > onset_threshold
        vel    = velocity_preds[:, key]

        note_start = None
        note_velocity = 0

        for t in range(T):
            if onset[t] and active[t]:
                # New note starting: close any open note first
                if note_start is not None:
                    notes.append({
                        'pitch': key + 21,  # Piano key 0 = MIDI pitch 21 (A0)
                        'start': note_start / frame_rate,
                        'end': t / frame_rate,
                        'velocity': int(note_velocity * 127)
                    })
                note_start = t
                note_velocity = vel[t]

            elif note_start is not None and not active[t]:
                # Note ended
                notes.append({
                    'pitch': key + 21,
                    'start': note_start / frame_rate,
                    'end': t / frame_rate,
                    'velocity': int(note_velocity * 127)
                })
                note_start = None

        # Close any note still open at end of chunk
        if note_start is not None:
            notes.append({
                'pitch': key + 21,
                'start': note_start / frame_rate,
                'end': T / frame_rate,
                'velocity': int(note_velocity * 127)
            })

    return sorted(notes, key=lambda n: n['start'])

def generate_and_compare(model, dataloader, device='cuda'):
    model.eval()

    # Grab one batch
    mel, frames, onsets, vels = next(iter(dataloader))

    # Send only the first item in the batch to the GPU for inference
    mel_input = mel[0:1].to(device)

    print("🎵 Starting Single-Pass Generation...")

    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            # One single forward pass processes the entire 16 seconds!
            onset_logits, frame_logits, velocity_pred = model(mel_input)

            # Apply Sigmoid to get probabilities [0, 1]
            onset_probs = torch.sigmoid(onset_logits).squeeze(0).cpu().float().numpy()
            frame_probs = torch.sigmoid(frame_logits).squeeze(0).cpu().float().numpy()
            vel_preds = velocity_pred.squeeze(0).cpu().float().numpy()
    # Get true targets for the first batch item, transpose to [512, 88]
    true_onsets = onsets[0].transpose(0, 1).numpy()
    true_frames = frames[0].transpose(0, 1).numpy()
    true_vels = vels[0].transpose(0, 1).numpy()

    # Convert binary rolls back to human-readable notes
    pred_notes = roll_to_notes(onset_probs, frame_probs, vel_preds)
    real_notes = roll_to_notes(true_onsets, true_frames, true_vels)

    print("\n--- Inference Complete ---")
    print(f"Total Real Notes: {len(real_notes)} | Total Predicted Notes: {len(pred_notes)}")
    print("\nFirst 15 Notes Comparison:")
    print(f"{'Real Pitch':<12} | {'Pred Pitch':<12} || {'Real Start (s)':<15} | {'Pred Start (s)':<15}")
    print("-" * 65)

    # Print a side-by-side comparison of the first 15 notes
    max_notes = min(15, max(len(real_notes), len(pred_notes)))

    for i in range(max_notes):
        r_pitch = str(real_notes[i]['pitch']) if i < len(real_notes) else "---"
        r_start = f"{real_notes[i]['start']:.2f}" if i < len(real_notes) else "---"

        p_pitch = str(pred_notes[i]['pitch']) if i < len(pred_notes) else "---"
        p_start = f"{pred_notes[i]['start']:.2f}" if i < len(pred_notes) else "---"

        pitch_match = "✅" if r_pitch == p_pitch else "❌"
        # For time, we allow a small +/- 0.05 second tolerance for a "match" visually
        time_match = "✅" if (r_start != "---" and p_start != "---" and abs(float(r_start) - float(p_start)) <= 0.05) else "❌"

        print(f"{r_pitch:<12} | {p_pitch:<10} {pitch_match} || {r_start:<15} | {p_start:<13} {time_match}")

    # Set model back to train mode for the next epoch
    model.train()

In [ ]:
from tqdm import tqdm
import torch
import torch.nn.functional as F
from torch.cuda.amp import GradScaler
import math

def get_cosine_lr_multiplier(step, total_steps, warmup_steps=1000):
    """Calculates a learning rate factor with linear warmup and cosine decay."""
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

def focal_loss(logits, targets, gamma=2.0, pos_weight=5.0):
    """
    Combines Focal Loss (for hard-example mining) with pos_weight
    (to prevent the model from collapsing to all-silence predictions).
    """
    # 1. Apply the critical positive weight directly to the base BCE
    # This forces the model to care 5x more about missing a note than guessing a false positive
    weight_tensor = torch.tensor([pos_weight], device=logits.device)
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction='none', pos_weight=weight_tensor
    )

    # 2. Calculate probabilities
    probs = torch.sigmoid(logits)

    # 3. p_t: probability of the correct class
    p_t = probs * targets + (1 - probs) * (1 - targets)

    # 4. Focal weight: scales down the loss for easy, confident predictions (like obvious silence)
    focal_weight = (1 - p_t) ** gamma

    # Apply focal weight and average
    loss = (focal_weight * bce).mean()

    return loss
def compute_loss(onset_logits, frame_logits, velocity_pred,
                 onset_target, frame_target, velocity_target):

    # Target shapes from dataloader are [Batch, 88, 512].
    # Logit shapes from model are [Batch, 512, 88]. We must transpose targets to match.
    onset_target = onset_target.transpose(1, 2)
    frame_target = frame_target.transpose(1, 2)
    velocity_target = velocity_target.transpose(1, 2)

    onset_loss = focal_loss(onset_logits, onset_target, gamma=2.0, pos_weight=5.0)
    frame_loss = focal_loss(frame_logits, frame_target, gamma=2.0, pos_weight=5.0)
    # Velocity loss is only calculated for frames where a note ACTUALLY starts
    onset_mask = onset_target.bool()
    if onset_mask.any():
        velocity_loss = F.mse_loss(velocity_pred[onset_mask], velocity_target[onset_mask])
    else:
        velocity_loss = torch.tensor(0.0, device=onset_logits.device)

    # Onsets are the hardest and most important part to get right, so we weight it 2x
    total_loss = 2.0 * onset_loss + frame_loss + 0.5 * velocity_loss

    return total_loss, onset_loss, frame_loss, velocity_loss

def run_frame_epoch(model, loader, test_loader=None, base_lr=3e-4, epochs=30, device='cuda', grad_accum_steps=4):
    model.to(device)

    freq_mask = T.FrequencyMasking(freq_mask_param=24).to(device)
    time_mask = T.TimeMasking(time_mask_param=40).to(device)

    # Using slightly higher weight decay (0.01) is standard for AdamW + BCE
    optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.01)

    # Scaler for safely handling bfloat16 mixed precision gradients
    scaler = torch.amp.GradScaler('cuda')

    global_step = 0
    # Adjust total steps for the scheduler based on gradient accumulation
    total_steps = (len(loader) // grad_accum_steps) * epochs

    for epoch in range(epochs):
        # -----------------------------
        # 1. TRAINING PHASE
        # -----------------------------
        model.train()
        total_epoch_loss = 0
        optimizer.zero_grad()

        loop = tqdm(enumerate(loader), total=len(loader), desc=f"Epoch {epoch+1} [Train]")

        for step, (mel, frames, onsets, vels) in loop:
            # Dynamically update LR
            lr_factor = get_cosine_lr_multiplier(global_step, total_steps, warmup_steps=1000)
            current_lr = base_lr * lr_factor
            for param_group in optimizer.param_groups:
                param_group['lr'] = current_lr

            # Extract inputs & send to GPU
            mel = mel.to(device, dtype=torch.bfloat16)  # [Batch, 1, 229, 512]
            frames = frames.to(device)                  # [Batch, 88, 512]
            onsets = onsets.to(device)                  # [Batch, 88, 512]
            vels = vels.to(device)                      # [Batch, 88, 512]

            # --- DYNAMIC GPU AUGMENTATION (Training Only) ---
            if model.training:
                # Random Gain
                gain_db = torch.empty((mel.size(0), 1, 1, 1), device=device, dtype=mel.dtype).uniform_(-6, 6)
                mel = mel + gain_db

                # Gaussian Noise
                mel = mel + torch.randn_like(mel) * 0.1

                # SpecAugment
                if torch.rand(1).item() > 0.3:
                    for _ in range(torch.randint(1, 3, (1,)).item()):
                        mel = freq_mask(mel)
                    for _ in range(torch.randint(1, 3, (1,)).item()):
                        mel = time_mask(mel)

            # Forward Pass with Autocast
            with torch.autocast(device_type=device, dtype=torch.bfloat16):
                onset_logits, frame_logits, velocity_pred = model(mel)

                loss, o_loss, f_loss, v_loss = compute_loss(
                    onset_logits, frame_logits, velocity_pred,
                    onsets, frames, vels
                )

                # Scale loss down by accumulation steps
                loss = loss / grad_accum_steps

            # Backward pass via Scaler
            scaler.scale(loss).backward()

            # Gradient Accumulation Step
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

                global_step += 1

            total_epoch_loss += (loss.item() * grad_accum_steps)

            if (step + 1) % 50 == 0:
                with torch.no_grad():
                    onset_preds = (torch.sigmoid(onset_logits) > 0.5).float()
                    onset_targets_t = onsets.transpose(1, 2)
                    acc = (onset_preds == onset_targets_t).float().mean()

                loop.set_postfix(
                    Loss=f"{(loss.item() * grad_accum_steps):.3f}",
                    On_BCE=f"{o_loss.item():.3f}",
                    Fr_BCE=f"{f_loss.item():.3f}",
                    On_Acc=f"{acc.item():.2%}"
                )

        print(f"\n--- Epoch {epoch+1} Train Complete | Avg Train Loss: {total_epoch_loss / len(loader):.4f} ---")

        # -----------------------------
        # 2. VALIDATION PHASE
        # -----------------------------
        if test_loader is not None:
            model.eval()  # Disable Dropout and BatchNorm updates
            val_loss = 0.0
            val_o_loss = 0.0
            val_f_loss = 0.0
            val_acc = 0.0

            # torch.no_grad() disables gradient tracking to save VRAM and compute
            with torch.no_grad():
                val_loop = tqdm(test_loader, total=len(test_loader), desc=f"Epoch {epoch+1} [Valid]")

                for mel, frames, onsets, vels in val_loop:
                    mel = mel.to(device, dtype=torch.bfloat16)
                    frames = frames.to(device)
                    onsets = onsets.to(device)
                    vels = vels.to(device)

                    # Notice: No gain, noise, or SpecAugment is applied here!

                    with torch.autocast(device_type=device, dtype=torch.bfloat16):
                        onset_logits, frame_logits, velocity_pred = model(mel)

                        loss, o_loss, f_loss, v_loss = compute_loss(
                            onset_logits, frame_logits, velocity_pred,
                            onsets, frames, vels
                        )

                    # Accumulate metrics
                    val_loss += loss.item()
                    val_o_loss += o_loss.item()
                    val_f_loss += f_loss.item()

                    onset_preds = (torch.sigmoid(onset_logits) > 0.5).float()
                    onset_targets_t = onsets.transpose(1, 2)
                    acc = (onset_preds == onset_targets_t).float().mean()
                    val_acc += acc.item()

            # Calculate Averages
            num_val_batches = len(test_loader)
            avg_val_loss = val_loss / num_val_batches

            print(f"📊 --- Validation Metrics ---")
            print(f"Val Total Loss: {avg_val_loss:.4f}")
            print(f"Val Onset Loss: {val_o_loss / num_val_batches:.4f}")
            print(f"Val Frame Loss: {val_f_loss / num_val_batches:.4f}")
            print(f"Val Onset Acc:  {val_acc / num_val_batches:.2%}")
            print("----------------------------\n")

        # -----------------------------
        # 3. GENERATE & COMPARE
        # -----------------------------
        # Uses whichever loader you pass to it to print the note alignment check
        generate_and_compare(model, test_loader if test_loader else loader)

In [ ]:
run_frame_epoch(model, loader, test_loader)

Epoch 1 [Train]:  63%|██████▎   | 3155/4973 [10:39<06:08,  4.94it/s, Fr_BCE=0.035, Loss=0.046, On_Acc=99.76%, On_BCE=0.004]


KeyboardInterrupt: 

In [ ]:
import os
import h5py
import torch
import random
import torchaudio
import note_seq
import pretty_midi
import numpy as np
import mir_eval

def roll_to_notes(onset_probs, frame_probs, velocity_preds, onset_threshold=0.55, frame_threshold=0.5, hop_length=512, sr=16000):
    """Converts 88-key binary piano rolls back into a list of MIDI note dictionaries."""
    frame_rate = sr / hop_length
    T = onset_probs.shape[0]
    notes = []

    for key in range(88):
        active = frame_probs[:, key] > frame_threshold
        onset  = onset_probs[:, key] > onset_threshold
        vel    = velocity_preds[:, key]

        note_start = None
        note_velocity = 0

        for t in range(T):
            if onset[t] and active[t]:
                if note_start is not None:
                    notes.append({
                        'pitch': key + 21,
                        'start': note_start / frame_rate,
                        'end': t / frame_rate,
                        'velocity': int(note_velocity * 127)
                    })
                note_start = t
                note_velocity = vel[t]

            elif note_start is not None and not active[t]:
                notes.append({
                    'pitch': key + 21,
                    'start': note_start / frame_rate,
                    'end': t / frame_rate,
                    'velocity': int(note_velocity * 127)
                })
                note_start = None

        if note_start is not None:
            notes.append({
                'pitch': key + 21,
                'start': note_start / frame_rate,
                'end': T / frame_rate,
                'velocity': int(note_velocity * 127)
            })

    return sorted(notes, key=lambda n: n['start'])

def save_notes_to_midi(notes, filename):
    """Saves a list of note dictionaries to a standard MIDI file."""
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)  # Acoustic Grand Piano
    for n in notes:
        # Clamp bounds just in case
        vel = max(0, min(127, n['velocity']))
        inst.notes.append(pretty_midi.Note(
            velocity=vel, pitch=n['pitch'], start=n['start'], end=n['end']
        ))
    pm.instruments.append(inst)
    pm.write(filename)

import os
import h5py
import torch
import random
import torchaudio
import note_seq
import pretty_midi
import numpy as np
import mir_eval
import math

# (Keep your existing roll_to_notes and save_notes_to_midi functions here)

def transcribe_full_random_song(model, h5_path, output_filename="full_transcription.mid", device='cuda'):
    model.eval()
    sample_rate = 16000
    chunk_duration_sec = 30.0  # Keep safe 30s chunks to respect VRAM and Positional Encoding limits

    # 1. Initialize Mel Spectrogram Transformer
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate, n_fft=2048, hop_length=512, n_mels=229, f_min=30.0, f_max=8000.0
    )

    # 2. Open HDF5 and pick a random track
    with h5py.File(h5_path, "r") as f:
        track_name = random.choice(list(f.keys()))
        full_audio = f[track_name]['audio'][:].astype('float32') / 32768.0

        raw_midi_bytes = f[track_name]['midi'][()].tobytes()
        gt_sequence = note_seq.NoteSequence.FromString(raw_midi_bytes)

    total_duration = len(full_audio) / sample_rate
    print(f"🎧 Selected Track: {track_name} | Total Duration: {total_duration:.1f} seconds")

    # 3. Extract Full Ground Truth Notes
    ref_intervals = []
    ref_pitches = []
    for note in gt_sequence.notes:
        if note.end_time > note.start_time:
            ref_intervals.append([note.start_time, note.end_time])
            ref_pitches.append(note.pitch)

    # Save Ground Truth MIDI
    gt_filename = output_filename.replace(".mid", "_actual.mid")
    note_seq.sequence_proto_to_midi_file(gt_sequence, gt_filename)
    print(f"✅ Full ground truth saved as {gt_filename}")

    # 4. CHUNK & STITCH INFERENCE
    all_pred_notes = []
    num_chunks = math.ceil(total_duration / chunk_duration_sec)

    print(f"🎵 Running Inference across {num_chunks} chunks...")

    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        for i in range(num_chunks):
            start_time = i * chunk_duration_sec
            end_time = min((i + 1) * chunk_duration_sec, total_duration)

            start_sample = int(start_time * sample_rate)
            end_sample = int(end_time * sample_rate)
            audio_chunk = full_audio[start_sample:end_sample]

            # Skip if the trailing chunk is completely empty
            if len(audio_chunk) == 0:
                continue

            # Convert to Log-Mel Spectrogram
            waveform = torch.tensor(audio_chunk).unsqueeze(0)
            mel_spec = mel_transform(waveform)
            log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-5)).unsqueeze(0).to(device, dtype=torch.bfloat16)

            # Instantaneous Forward Pass for this 30s chunk
            onset_logits, frame_logits, velocity_pred = model(log_mel_spec)

            # Apply Sigmoids and cast to float32
            onset_probs = torch.sigmoid(onset_logits).squeeze(0).cpu().float().numpy()
            frame_probs = torch.sigmoid(frame_logits).squeeze(0).cpu().float().numpy()
            vel_preds = velocity_pred.squeeze(0).cpu().float().numpy()

            # Decode to note dictionaries
            chunk_notes = roll_to_notes(onset_probs, frame_probs, vel_preds, sr=sample_rate)

            # STITCHING: Shift the start and end times by the chunk's absolute start time
            for n in chunk_notes:
                n['start'] += start_time
                n['end'] += start_time
                all_pred_notes.append(n)

    print("🎹 Stitching complete. Saving to MIDI...")
    save_notes_to_midi(all_pred_notes, output_filename)
    print(f"✅ Full prediction saved as {output_filename}")

    # 5. Evaluate the ENTIRE song with mir_eval
    if ref_intervals and all_pred_notes:
        ref_intervals = np.array(ref_intervals)
        ref_pitches = np.array(ref_pitches)

        est_intervals = np.array([[n['start'], n['end']] for n in all_pred_notes])
        est_pitches = np.array([n['pitch'] for n in all_pred_notes])

        if len(est_intervals) > 0:
            p, r, f1, _ = mir_eval.transcription.precision_recall_f1_overlap(
                ref_intervals,
                mir_eval.util.midi_to_hz(ref_pitches),
                est_intervals,
                mir_eval.util.midi_to_hz(est_pitches),
                offset_ratio=None,
                onset_tolerance=0.05
            )
        else:
            p, r, f1 = 0.0, 0.0, 0.0

        print("\n📊 --- FULL SONG TRANSCRIPTION ACCURACY (50ms Onset Tolerance) ---")
        print(f"Actual Notes:    {len(ref_pitches)}")
        print(f"Predicted Notes: {len(est_pitches)}")
        print("-------------------------------------------------------")
        print(f"Precision: {p:.2%}")
        print(f"Recall:    {r:.2%}")
        print(f"F1-Score:  {f1:.2%}")
        print("-------------------------------------------------------\n")
    else:
        print("\n⚠️ Not enough notes in sequence to calculate F1 score.")

    model.train()
    return output_filename

In [ ]:
from IPython.display import FileLink

# Pass your actual dataset path here
output_file = transcribe_full_random_song(
    model=model,
    h5_path='/kaggle/input/maestrov300-hdf5/2018.h5', # Use your exact filename!
    output_filename='/home/random_transcription1.mid',
)

# Display the download link below the cell
FileLink(output_file)

🎧 Selected Track: MIDI-Unprocessed_Recital9-11_MID--AUDIO_11_R1_2018_wav--2 | Total Duration: 1196.2 seconds
✅ Full ground truth saved as /home/random_transcription1_actual.mid
🎵 Running Inference across 40 chunks...
🎹 Stitching complete. Saving to MIDI...
✅ Full prediction saved as /home/random_transcription1.mid

📊 --- FULL SONG TRANSCRIPTION ACCURACY (50ms Onset Tolerance) ---
Actual Notes:    9525
Predicted Notes: 11134
-------------------------------------------------------
Precision: 59.13%
Recall:    69.12%
F1-Score:  63.74%
-------------------------------------------------------



/home/random_transcription1.mid

In [ ]:
save_path = '/home/conformer_piano_final_weights.pt'

torch.save(model, save_path)
print(f"💾 Final model weights saved to {save_path}")